Pivot

In [0]:
SELECT customer_id,
       address_type,
       address_line_1,
       city,
       state,
       postcode
FROM gizmobox.bronze.v_addresses;



In [0]:
SELECT * from gizmobox.bronze.v

JSON
1) extract top level column value
2) extract array element
3) extract nested column
4) CAST column value

In [0]:
SELECT * from gizmobox.bronze.v_orders;

extract top level

In [0]:
SELECT value:order_id as order_id,
value from gizmobox.bronze.v_orders;

extract array

In [0]:
SELECT value:items as items,
value from gizmobox.bronze.v_orders;

In [0]:
SELECT value:items[0] as items,
value from gizmobox.bronze.v_orders;

In [0]:
SELECT value:items[0] as items_1,
       value:items[1] as items_2,

value from gizmobox.bronze.v_orders;

nested columns

In [0]:
SELECT value:items[0].item_id as items_id_items,
      value:items[0] as items_1,
       value:items[1] as items_2,

value from gizmobox.bronze.v_orders;

In [0]:
SELECT value:items[0].item_id::INTEGER as items_id_items,
      value:items[0] as items_1,
       value:items[1] as items_2,

value from gizmobox.bronze.v_orders;

PRE-PROCESS the JSON string to fix data quality issue

In [0]:
SELECT value,
       regexp_replace(
           value,
           '"order_date":(\\d{4}-\\d{2}-\\d{2})',
           '"order_date":"\\1"'
       ) AS fixed_value
FROM gizmobox.bronze.v_orders;


In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_order_fixed AS
SELECT value,
       regexp_replace(
           value,
           '"order_date":(\\d{4}-\\d{2}-\\d{2})',
           '"order_date":"\\1"'
       ) AS fixed_value
FROM gizmobox.bronze.v_orders;

In [0]:
SELECT fixed_value from tv_order_fixed;

convert jason string to json object

In [0]:
SELECT schema_of_json(fixed_value)
FROM tv_order_fixed
LIMIT 1;

In [0]:
SELECT from_json(fixed_value,
        'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
FROM tv_order_fixed
LIMIT 1;

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.json_orders 
AS
SELECT from_json(fixed_value,
        'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
FROM tv_order_fixed
LIMIT 1;

In [0]:
SELECT * FROM gizmobox.silver.json_orders;

In [0]:
SELECT json_value.order_id,
       json_value.order_date,
       json_value.order_status,
       json_value.payment_method,
       json_value.total_amount
       
 FROM gizmobox.silver.json_orders;